# NAVROS — benchmark en Colab TPU v5e-1

1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → v5e-1 TPU** (o v6e-1).
2. **Ejecutar todo**. Tarda ~15–25 min (incluye instalar PyTorch/XLA si falta y compilar).
3. Copia las líneas `RESULTADO` del final (o haz una captura) y compártelas.

No escribe nada en tu Drive ni necesita credenciales.

In [ ]:
# 1) Entorno: ¿hay PyTorch/XLA? Si no, se instala una versión emparejada con torch.
import importlib.util, os, subprocess, sys
os.environ["PJRT_DEVICE"] = "TPU"
def sh(c):
    print("$", c, flush=True)
    r = subprocess.run(c, shell=True, text=True, capture_output=True)
    print((r.stdout or "")[-1500:], (r.stderr or "")[-1500:], flush=True)
    return r.returncode
pl = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True).stdout
print("\n".join(l for l in pl.splitlines() if any(s in l.lower() for s in ("torch", "jax", "xla", "libtpu"))))
LIBTPU = "-f https://storage.googleapis.com/libtpu-releases/index.html -f https://storage.googleapis.com/libtpu-wheels/index.html"
if importlib.util.find_spec("torch_xla") is None:
    import importlib.metadata as md
    tv = md.version("torch").split("+")[0] if importlib.util.find_spec("torch") else None
    ok = tv is not None and sh(f'{sys.executable} -m pip install -q "torch_xla[tpu]=={tv}" {LIBTPU}') == 0
    for v in ([] if ok else ["2.9.0", "2.8.0", "2.7.0"]):
        if sh(f'{sys.executable} -m pip install -q "torch=={v}" --index-url https://download.pytorch.org/whl/cpu') == 0 and \
           sh(f'{sys.executable} -m pip install -q "torch_xla[tpu]=={v}" {LIBTPU}') == 0:
            ok = True
            break
    print("torch_xla instalado:", ok)
print(subprocess.run([sys.executable, "-c", "import torch, torch_xla; print('torch', torch.__version__, 'torch_xla', torch_xla.__version__)"],
                     capture_output=True, text=True))

In [ ]:
# 2) Código de NAVROS (repo público, commit fijado)
COMMIT = "de544499aa50bf4ebdf2ef3dc4171f137c7836e2"
if not os.path.exists("/content/navros-ai"):
    sh("git clone -q https://github.com/navroscol/navros-ai /content/navros-ai")
sh(f"git -C /content/navros-ai fetch -q && git -C /content/navros-ai checkout -q {COMMIT}")

In [ ]:
# 3) Benchmark: cada configuración en su propio proceso (un fallo de memoria no tumba las demás)
import json
base = dict(device="xla", precision="bf16", spmd=False, muon_buf="bf16", grad_ckpt=True, steps=6, warmup=3)
configs = [dict(preset="tiny", T=256, micro=8, steps=3, warmup=2, grad_ckpt=False),
           dict(preset="navros-1b", T=1024, micro=4),
           dict(preset="navros-1b-fix", T=1024, micro=4),
           dict(preset="navros-1b", T=1024, micro=8)]
resultados = []
for c in configs:
    kw = base | c
    print("=" * 20, kw, flush=True)
    p = subprocess.run([sys.executable, "/content/navros-ai/scripts/06_bench.py", "--json", json.dumps(kw)],
                       capture_output=True, text=True, env=dict(os.environ, PJRT_DEVICE="TPU"))
    print((p.stdout or "")[-2500:], (p.stderr or "")[-1500:], flush=True)
    resultados += [l for l in (p.stdout or "").splitlines() if l.startswith("RESULTADO")]
print("\n\n".join(resultados) or "sin resultados")